# Module 6: Advanced Producer

本章目标：
- 使用消息压缩减少网络和存储开销
- 实现自定义 Partitioner 实现精确路由
- 使用幂等 Producer 防止重复写入
- 使用事务 Producer 实现原子性批量写入
- 理解 Producer 批量发送（batch）与延迟（linger）的权衡

---

## 前置条件

- 集群运行中：`docker compose up -d`
- 已安装依赖：`uv sync`

In [ ]:
import asyncio, time, json, hashlib
from aiokafka import AIOKafkaProducer, AIOKafkaConsumer
from aiokafka.admin import AIOKafkaAdminClient, NewTopic

BROKERS = "localhost:19094,localhost:29094,localhost:39094"
TOPIC = "advanced-producer-demo"

async def ensure_topic(brokers, topic, num_partitions=4, replication_factor=3):
    admin = AIOKafkaAdminClient(bootstrap_servers=brokers)
    await admin.start()
    try:
        existing = await admin.list_topics()
        if topic not in existing:
            await admin.create_topics([
                NewTopic(name=topic, num_partitions=num_partitions, replication_factor=replication_factor)
            ])
            print(f"✓ Topic '{topic}' created")
        else:
            print(f"Topic '{topic}' already exists")
    finally:
        await admin.close()

await ensure_topic(BROKERS, TOPIC)

## 1. 消息压缩

Producer 端压缩，Consumer 端透明解压。适合文本类（JSON、日志）数据，可大幅节省带宽。

| 压缩算法 | 压缩率 | CPU 开销 | 适用场景 |
|---------|--------|---------|----------|
| none | - | 最低 | 低延迟、小消息 |
| gzip | 最高 | 高 | 带宽敏感，对延迟不敏感 |
| snappy | 中 | 低 | 均衡选择 |
| lz4 | 中偏低 | 极低 | 高吞吐场景 |
| zstd | 高 | 中 | 推荐默认（Kafka 2.1+）|

In [ ]:
# 生成一段 JSON 消息（文本压缩效果好）
def make_json_message(i):
    data = {
        "event_id": f"evt-{i:06d}",
        "user_id": f"user-{i % 100}",
        "action": "page_view",
        "url": f"/products/category/electronics/item-{i}",
        "timestamp": int(time.time() * 1000),
        "metadata": {"session": "abc123", "device": "mobile", "country": "CN"},
    }
    return json.dumps(data).encode()

async def benchmark_compression(brokers, topic, compression, count=100):
    producer = AIOKafkaProducer(
        bootstrap_servers=brokers,
        compression_type=compression,
        acks="all",
    )
    await producer.start()
    start = time.perf_counter()
    try:
        for i in range(count):
            await producer.send_and_wait(topic, make_json_message(i))
    finally:
        await producer.stop()
    elapsed = time.perf_counter() - start
    print(f"  compression={compression or 'none':8s} | {count} msgs | {elapsed*1000:.0f} ms")

print("=== Compression benchmark ===")
for codec in [None, "gzip", "snappy", "lz4", "zstd"]:
    await benchmark_compression(BROKERS, TOPIC, codec)

## 2. 自定义 Partitioner

业务场景：不同地区的消息路由到专用分区，便于按地区消费。

In [ ]:
from aiokafka.partitioner import DefaultPartitioner

REGION_PARTITION_MAP = {
    b"cn": 0,
    b"us": 1,
    b"eu": 2,
}

class RegionPartitioner(DefaultPartitioner):
    """将消息按地区 key 路由到固定分区；未知地区回退到默认哈希。"""

    def __call__(self, key, all_partitions, available):
        if key in REGION_PARTITION_MAP:
            p = REGION_PARTITION_MAP[key]
            if p in all_partitions:
                return p
        return super().__call__(key, all_partitions, available)

async def region_partitioner_demo(brokers, topic):
    producer = AIOKafkaProducer(
        bootstrap_servers=brokers,
        partitioner=RegionPartitioner(),
    )
    await producer.start()
    try:
        events = [("cn", "北京用户登录"), ("us", "NY purchase"), ("eu", "Berlin checkout"),
                  ("cn", "上海浏览"), ("jp", "Tokyo search")]  # jp 回退到默认
        for region, msg in events:
            meta = await producer.send_and_wait(
                topic, key=region.encode(), value=msg.encode()
            )
            print(f"  region={region!r:4} -> partition={meta.partition}: {msg}")
    finally:
        await producer.stop()

print("=== Custom Partitioner ===")
await region_partitioner_demo(BROKERS, TOPIC)

## 3. 批量发送优化：linger_ms 与 batch_size

In [ ]:
async def batch_benchmark(brokers, topic, linger_ms, batch_size, count=500):
    producer = AIOKafkaProducer(
        bootstrap_servers=brokers,
        linger_ms=linger_ms,           # 等待 batch 填充的最大时间
        max_batch_size=batch_size,     # 单批次最大字节数
        acks=1,
    )
    await producer.start()
    start = time.perf_counter()
    try:
        tasks = [
            producer.send(topic, f"batch-msg-{i}".encode())
            for i in range(count)
        ]
        futures = await asyncio.gather(*tasks)
        await producer.flush()  # 等待所有消息发出
    finally:
        await producer.stop()
    elapsed = time.perf_counter() - start
    print(f"  linger={linger_ms:3d}ms batch={batch_size//1024:4d}KB | {count} msgs | {elapsed*1000:.0f}ms | {count/elapsed:.0f} msg/s")

print("=== Batch tuning ===")
print(f"  {'config':35s} | msgs | total | throughput")
for linger, batch in [(0, 16*1024), (5, 64*1024), (20, 256*1024)]:
    await batch_benchmark(BROKERS, TOPIC, linger, batch)

## 4. 事务 Producer：原子性批量写入

事务保证：多条消息要么全部成功写入，要么全部不可见（Kafka 的原子写入语义）。

典型场景：将处理结果同时写入多个 Topic，需要保证一致性。

In [ ]:
TOPIC_A = "tx-output-a"
TOPIC_B = "tx-output-b"

async def setup_tx_topics(brokers):
    admin = AIOKafkaAdminClient(bootstrap_servers=brokers)
    await admin.start()
    try:
        existing = await admin.list_topics()
        new_topics = [
            NewTopic(name=t, num_partitions=1, replication_factor=3)
            for t in [TOPIC_A, TOPIC_B] if t not in existing
        ]
        if new_topics:
            await admin.create_topics(new_topics)
            print(f"✓ Created: {[t.name for t in new_topics]}")
    finally:
        await admin.close()

async def transactional_write(brokers, commit=True):
    producer = AIOKafkaProducer(
        bootstrap_servers=brokers,
        transactional_id="demo-tx-producer",  # 唯一事务 ID
        enable_idempotence=True,
        acks="all",
    )
    await producer.start()
    try:
        await producer.begin_transaction()
        await producer.send_and_wait(TOPIC_A, b"order-confirmed")
        await producer.send_and_wait(TOPIC_B, b"inventory-deducted")
        if commit:
            await producer.commit_transaction()
            print("  ✓ Transaction committed: both messages visible")
        else:
            await producer.abort_transaction()
            print("  ✗ Transaction aborted: neither message visible")
    except Exception as e:
        await producer.abort_transaction()
        print(f"  ✗ Error, transaction aborted: {e}")
        raise
    finally:
        await producer.stop()

await setup_tx_topics(BROKERS)
print("\n=== Committed transaction ===")
await transactional_write(BROKERS, commit=True)
print("\n=== Aborted transaction ===")
await transactional_write(BROKERS, commit=False)

## 总结

| 功能 | 参数 | 适用场景 |
|------|------|----------|
| 压缩 | `compression_type='zstd'` | 大量文本/JSON 消息，节省带宽 |
| 自定义分区 | `partitioner=MyPartitioner()` | 业务路由、数据亲和性 |
| 批量优化 | `linger_ms`, `max_batch_size` | 高吞吐写入 |
| 幂等 | `enable_idempotence=True` | 防止重试导致重复消息 |
| 事务 | `transactional_id='...'` | 跨 Topic 原子写入 |

## 下一章

**Module 7: Advanced Consumer** — 手动 offset 提交、seek 到指定位置、精确一次语义（EOS）。